# 找到伺服器


In [1]:
import re
import time
import uiautomator2 as u2
import img_tools
device = u2.connect('emulator-5560')

 # 定位伺服器

In [ ]:

# =================設定區域=================
#在此輸入你想點擊的技能 ID
target_id = "S1470" 

# 裁切偏移量 (對應 img[177:542, 127:190])
OFFSET_Y = 177
OFFSET_X = 127
# =========================================

# 1. 截圖與辨識
print(f"正在尋找技能: {target_id} ...")
img = device.screenshot(format='opencv')[177:542, 127:190]
results = img_tools.analyze_skill_via_http(img)['ocr_results']

# 標記是否找到，用於最後的回報
found_target = False

for res in results:
    text = res['text']
    
    # 2. 解析 ID (抗雜訊 Regex)
    match = re.search(r'(S\d+)', text)
    
    if match:
        skill_id = match.group(1)
        
        # 3. 比對是否為目標 ID
        if skill_id == target_id:
            bbox = res['bbox']
            
            # 4. 計算全域中心座標
            rel_center_x = (bbox[0] + bbox[2]) // 2
            rel_center_y = (bbox[1] + bbox[3]) // 2
            
            global_x = rel_center_x + OFFSET_X
            global_y = rel_center_y + OFFSET_Y
            
            print(f"✅ 找到目標 [{skill_id}]")
            print(f"   └─ 執行點擊座標: ({global_x}, {global_y})")
            
            # 5. 執行點擊
            device.click(global_x+300, global_y)
            
            found_target = True
            break  # 找到後就跳出迴圈，不用繼續找了

if not found_target:
    print(f"❌ 畫面中未發現技能 [{target_id}]")

# 精準車位

In [ ]:
import re
import time

# ================= 設定區域 =================
target_num = "5"   # 在這裡輸入你想點的車位號碼 (1, 5, 8, 9...)

# 1. 裁切偏移量 (必須加回去，座標才會對)
OFFSET_Y = 177
OFFSET_X = 119

# 2. 設定「右側按鈕」的 X 座標
# 因為你的裁切只到 265，但按鈕在更右邊 (參考全圖大概在 580 左右)
# 你可以根據實際情況調整這個數字
BUTTON_X_GLOBAL = 480 

# 3. Y 軸微調 (如果發現點太高或太低，可以改這裡，正數往下，負數往上)
Y_ADJUST = 10 
# ===========================================

# 截圖與辨識 (只截取文字列表部分)
img = device.screenshot(format='opencv')[177:542, 119:265]
results = img_tools.analyze_skill_via_http(img)['ocr_results']

print(f"正在搜尋: 跨界車位{target_num} ...")

found = False
for res in results:
    text = res['text']
    
    # 搜尋 "跨界車位" + 你的數字
    # 使用 in 判斷，因為 OCR 有時會辨識成 "跨界車位1 " (後面有空白)
    if f"跨界車位{target_num}" in text:
        
        bbox = res['bbox'] # 相對座標 [x1, y1, x2, y2]
        
        # --- 計算座標 ---
        
        # 1. 算出文字的高度中心 (Y軸)
        rel_center_y = (bbox[1] + bbox[3]) // 2
        global_y = rel_center_y + OFFSET_Y + Y_ADJUST
        
        # 2. X軸使用我們設定好的固定位置 (點擊右邊按鈕)
        global_x = BUTTON_X_GLOBAL
        
        print(f"✅ 找到 [跨界車位{target_num}]")
        print(f"   └─ 文字位置(相對): {bbox}")
        print(f"   └─ 執行點擊座標: ({global_x}, {global_y})")
        
        # 執行點擊
        device.click(global_x, global_y)
        found = True
        break

if not found:
    print(f"❌ 沒找到 [跨界車位{target_num}]，請確認該車位是否在畫面中")
    #往下滑動一點
    device.swipe_ext("up", scale=0.3)
    time.sleep(0.1)
    device.click(267,482)
    #再次嘗試點擊-未寫
    

正在搜尋: 跨界車位5 ...
✅ 找到 [跨界車位5]
   └─ 文字位置(相對): [0, 153, 89, 182]
   └─ 執行點擊座標: (480, 354)


打車 -未寫驗證

In [2]:
import numpy as np
import time
import random
import uiautomator2 as u2
device =  u2.connect('7fe98fc6')   
d = device

# points 格式：[(x, y, (c1, c2, c3)), ...]
# 注意：d.screenshot(format='opencv') 回來的 img 是 OpenCV 的 BGR 順序
def _score_points(img, points, tol=18):
    coords = np.array([(x, y) for x, y, _ in points], dtype=np.int32)
    target = np.array([c for _, _, c in points], dtype=np.int16)

    xs = coords[:, 0]
    ys = coords[:, 1]

    # 取像素：img[y, x]，回傳 shape=(N,3)
    pix = img[ys, xs].astype(np.int16)

    diff = np.abs(pix - target)                    # (N,3)
    ok = (diff <= tol).all(axis=1)                 # (N,)
    hit_rate = ok.mean()
    mean_err = diff.mean()

    return hit_rate, mean_err

def detect_state(img, points1, points2, tol=18, min_hit=0.7):
    h1, e1 = _score_points(img, points1, tol=tol)
    h2, e2 = _score_points(img, points2, tol=tol)

    # 先比命中率，再比平均誤差（更穩）
    if (h1 < min_hit) and (h2 < min_hit):
        return "unknown", {"state1": (h1, e1), "state2": (h2, e2)}
    if h1 != h2:
        return ("state1" if h1 > h2 else "state2"), {"state1": (h1, e1), "state2": (h2, e2)}
    return ("state1" if e1 < e2 else "state2"), {"state1": (h1, e1), "state2": (h2, e2)}


# ====== 你的點（照你貼的結果直接放）======
points_state1 = [
    (161, 325, (204, 230, 236)),
    (341, 449, (204, 230, 236)),
    (134, 432, (204, 230, 236)),
    (78, 742, (175, 204, 213)),
    (448, 743, (158, 185, 205)),
    (13, 444, (103, 105, 105)),
    (10, 564, (103, 105, 105)),
    (230, 593, (204, 230, 236)),
    (121, 248, (204, 230, 236)),
    (88, 273, (204, 230, 236)), (369, 264, (204, 230, 236)), (274, 252, (204, 230, 236)), (334, 276, (204, 230, 236)), (93, 297, (203, 229, 235)), (93, 324, (204, 230, 236)), (86, 235, (204, 230, 236))
]

points_state2 = [
    (112, 387, (173, 203, 214)),
    (128, 449, (174, 204, 215)),
    (90, 648, (106, 118, 122)),
    (97, 734, (91, 105, 111)),
    (22, 730, (48, 48, 48)),
    (20, 586, (55, 55, 55)),
    (27, 396, (57, 57, 57)),
    (393, 200, (100, 116, 122)),
    (524, 748, (46, 46, 46)),
    (402, 770, (90, 106, 112)),
    (516, 895, (8, 13, 16)),
]
# d.click(198,790)
# time.sleep(0.5)
# ====== 使用 ======
# s1639-8+17:53:48

user_input = "17:53:48"
# 將使用者輸入的時間轉成time.struct_time
# sleep_time = time.strptime(user_input, "%H:%M:%S")
# #提早1.5秒開始 需要將使用者輸入的時間-1.5 
# sleep_time = time.localtime(time.mktime(sleep_time) - 1.5)   
# while True:
#     now_time = time.strftime("%H:%M:%S", time.localtime())
#     if now_time == time.strftime("%H:%M:%S", sleep_time):
#         break
time.sleep(2)
while(True):
    start_time = time.time()    
    img = d.screenshot(format='opencv')
    state, debug = detect_state(img, points_state1, points_state2, tol=18, min_hit=0.7)
    print(state)
    if state == "state1":
        #執行加入搶占
        d.click(random.randint(244,336),random.randint(730,751))
        d.click(random.randint(323,435),random.randint(544,566))
        time.sleep(0.1)
    elif state == "state2":
        #執行確定
        d.click(random.randint(323,435),random.randint(544,566))
        time.sleep(0.1)
    else:
        break
    print("偵測時間:", time.time() - start_time)

unknown
